In [ ]:
from google.colab import files
uploaded = files.upload()

Saving produtos_raw.csv to produtos_raw.csv


In [ ]:
import pandas as pd

# Se for Excel
df = pd.read_csv("produtos_raw.csv")

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
print("Tamanho:", df.shape)
print("\nColunas:", df.columns.tolist())
print("\nTipos de dados:")
print(df.dtypes)
print("\nPrimeiras linhas:")
df.head()

Tamanho: (157, 4)

Colunas: ['name', 'price', 'code', 'actual_category']

Tipos de dados:
name               object
price              object
code                int64
actual_category    object
dtype: object

Primeiras linhas:


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz


In [ ]:
# Clean 'price' column
df['price'] = df['price'].astype(str).str.replace('R$', '', regex=False)
df['price'] = df['price'].str.strip()
df['price'] = pd.to_numeric(df['price'])

# Re-apply category standardization (if not already done or if original was reverted)
import unicodedata

def clean_category(text):
    if isinstance(text, str):
        text = text.lower()
        text = ' '.join(text.split())
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    return text

df['actual_category_cleaned'] = df['actual_category'].apply(clean_category)

category_mapping = {
    'eletronicos': 'eletronicos',
    'eletronicoz': 'eletronicos',
    'eletrunicos': 'eletronicos',
    'e l e t r o n i c o s': 'eletronicos',
    'eletroniscos': 'eletronicos',
    'ancoragem': 'ancoragem',
    'ancoraguem': 'ancoragem',
    'ancorajm': 'ancoragem',
    'a n c o r a g e m': 'ancoragem',
    'encoragem': 'ancoragem',
    'ancorajem': 'ancoragem',
    'encoragi': 'ancoragem',
    'ancoragen': 'ancoragem',
    'ancorajen': 'ancoragem',
    'propulsao': 'propulsao',
    'propulcao': 'propulsao',
    'propucao': 'propulsao',
    'propulssao': 'propulsao',
    'prop': 'propulsao',
    'p r o p u l s a o': 'propulsao',
    'propulsam': 'propulsao'
}

df['actual_category'] = df['actual_category_cleaned'].replace(category_mapping)

# Drop intermediate cleaning columns
df = df.drop(columns=['actual_category_cleaned'])

print('\nDataFrame dtypes after cleaning:')
print(df.dtypes)

print('\nDataFrame head after cleaning:')
display(df.head())


DataFrame dtypes after cleaning:
name                object
price              float64
code                 int64
actual_category     object
dtype: object

DataFrame head after cleaning:


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,"33,122.52",1,eletronicos
1,Transponder Furuno Marlin,"13,998.15",2,eletronicos
2,Radar Furuno Pulse Leviathan,"9,024.19",3,eletronicos
3,Rádio AIS Hydro Tidal Zen,"3,381.88",4,eletronicos
4,Piloto Automático Furuno Storm,"23,669.01",5,eletronicos


In [ ]:
print("Valores nulos:")
print(df.isnull().sum())

print("\nDuplicatas:", df.duplicated().sum())

Valores nulos:
name               0
price              0
code               0
actual_category    0
dtype: int64

Duplicatas: 7


In [ ]:
duplicatas = df[df.duplicated(keep=False)]
print("Total de linhas duplicadas:", len(duplicatas))
duplicatas

Total de linhas duplicadas: 11


,name,price,code,actual_category
36,GPS Lowrance Evo Storm Drift,"6,067.71",37,eletronicos
37,GPS Lowrance Evo Storm Drift,"6,067.71",37,eletronicos
62,Motor Diesel Yanmar Velocity 37HP,"102,221.97",62,propulsao
63,Motor Diesel Yanmar Velocity 37HP,"102,221.97",62,propulsao
64,Motor Diesel Yanmar Velocity 37HP,"102,221.97",62,propulsao
65,Motor Diesel Yanmar Velocity 37HP,"102,221.97",62,propulsao
124,Boia de Arqueamento Delta Nexus,"4,349.86",145,ancoragem
131,Cabo de Nylon Delta Velocity Core Mako,"1,549.35",127,ancoragem
132,Cabo de Nylon Delta Velocity Core Mako,"1,549.35",127,ancoragem
150,Boia de Arqueamento Delta Nexus,"4,349.86",145,ancoragem


In [ ]:
# Remove linhas duplicadas e mantém a primeira ocorrência
df_limpo = df.drop_duplicates()

# Para aplicar a alteração diretamente no DataFrame original sem criar um novo:
df.drop_duplicates(inplace=True)


In [ ]:
print("Valores nulos:")
print(df.isnull().sum())

print("\nDuplicatas:", df.duplicated().sum())

Valores nulos:
name               0
price              0
code               0
actual_category    0
dtype: int64

Duplicatas: 0


In [ ]:
resumo_code = pd.DataFrame({
    "Métrica": [
        "Total de registros",
        "Códigos únicos",
        "Códigos nulos",
        "Duplicatas"
    ],
    "Valor": [
        len(df),
        df["code"].nunique(),
        df["code"].isnull().sum(),
        df["code"].duplicated().sum()
    ]
})
print("TABELA — CODE")
print(resumo_code.to_string(index=False))

🔢 TABELA — CODE
           Métrica  Valor
Total de registros    150
    Códigos únicos    150
     Códigos nulos      0
        Duplicatas      0


In [ ]:
resumo_name = pd.DataFrame({
    "Métrica": [
        "Total de registros",
        "Nomes únicos",
        "Nomes nulos",
        "Nomes em branco"
    ],
    "Valor": [
        len(df),
        df["name"].nunique(),
        df["name"].isnull().sum(),
        (df["name"].str.strip() == "").sum()
    ]
})
print("TABELA — NAME")
print(resumo_name.to_string(index=False))

📦 TABELA — NAME
           Métrica  Valor
Total de registros    150
      Nomes únicos    150
       Nomes nulos      0
   Nomes em branco      0


In [ ]:
resumo_price = pd.DataFrame({
    "Métrica": [
        "Total de registros",
        "Preço mínimo",
        "Preço máximo",
        "Média",
        "Zeros",
        "Negativos",
        "Nulos"
    ],
    "Valor": [
        len(df),
        f"R$ {df['price'].min():,.2f}",
        f"R$ {df['price'].max():,.2f}",
        f"R$ {df['price'].mean():,.2f}",
        (df["price"] == 0).sum(),
        (df["price"] < 0).sum(),
        df["price"].isnull().sum()
    ]
})
print("TABELA — PRICE")
print(resumo_price.to_string(index=False))

💰 TABELA — PRICE
           Métrica         Valor
Total de registros           150
      Preço mínimo     R$ 309.54
      Preço máximo R$ 148,198.23
             Média  R$ 34,683.82
             Zeros             0
         Negativos             0
             Nulos             0


In [ ]:
resumo_category = pd.DataFrame({
    "Métrica": [
        "Total de registros",
        "Categorias únicas",
        "Nulos",
        "Em branco"
    ],
    "Valor": [
        len(df),
        df["actual_category"].nunique(),
        df["actual_category"].isnull().sum(),
        (df["actual_category"].str.strip() == "").sum()
    ]
})
print("TABELA — ACTUAL_CATEGORY")
print(resumo_category.to_string(index=False))

# Ver quais são as categorias existentes
print("\nCategorias encontradas:")
print(df["actual_category"].value_counts())

🏷️ TABELA — ACTUAL_CATEGORY
           Métrica  Valor
Total de registros    150
 Categorias únicas      3
             Nulos      0
         Em branco      0

Categorias encontradas:
actual_category
eletronicos    50
propulsao      50
ancoragem      50
Name: count, dtype: int64


In [ ]:
# name — padronizar texto
df["name"] = df["name"].str.strip().str.title()

# actual_category — padronizar texto e preencher nulos
df["actual_category"] = df["actual_category"].str.strip().str.title()
df["actual_category"] = df["actual_category"].fillna("Sem Categoria")

# price — garantir tipo numérico e remover inválidos
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df[df["price"] > 0]

# code — garantir que é único
df = df.drop_duplicates(subset=["code"])

print("Tratamento concluído!")
print("Registros finais:", len(df))

✅ Tratamento concluído!
Registros finais: 150


In [ ]:
df.to_csv("produtos_limpo.csv", index=False)

✅ Base de produtos limpa salva!
